# P11 · Genuine VLA integration exercise: SmolVLA
**Execution status: not run or validated in this delivery.** The browser baseline is already runnable; this notebook is a separate GPU extension. It must not be described as a completed robot-policy evaluation.

Use Google Colab with a GPU, or a suitable local PyTorch environment. The pretrained base is intended for adaptation to a compatible robot/domain. A returned vector is not evidence that it can safely control your robot. This notebook has no hardware-control code.

Sources: [official model card](https://huggingface.co/lerobot/smolvla_base), [LeRobot](https://github.com/huggingface/lerobot), [SmolVLA paper](https://arxiv.org/abs/2506.01844). The checkpoint and dependency versions below were resolved when this artifact was prepared. No sample actions are fabricated.

In [ ]:
%pip install "lerobot[smolvla]==0.6.1"

In [ ]:
import torch, numpy as np, json, time, platform, importlib.metadata
from pathlib import Path
from huggingface_hub import snapshot_download
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors
MODEL_ID = "lerobot/smolvla_base"
REVISION = '5e8d12a6e2975b0e5e5fce7c8caf47c371d257b6'
checkpoint = snapshot_download(MODEL_ID, revision=REVISION)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
policy = SmolVLAPolicy.from_pretrained(checkpoint).to(device).eval()
preprocess, postprocess = make_pre_post_processors(policy.config, checkpoint,
    preprocessor_overrides={"device_processor": {"device": device}})
print("Required features:", policy.config.input_features)
print("Output features:", policy.config.output_features)
print("Action semantics must be checked against the chosen robot/dataset.")

## Supply a compatible observation, not a random photograph
Create `observation.npz` from a robot dataset compatible with the checkpoint. Include `observation.state` and every camera key printed above. Images must be RGB arrays, H×W×3, uint8. State must contain the checkpoint's expected features in the correct order and physical units. Also set a truthful instruction.

Do not fill missing cameras or state with zeros to imply a valid experiment. First confirm the dataset/checkpoint embodiment and normalization. If the contract differs, stop and write an explicit adapter rather than silently renaming arbitrary fields. In Colab, you can upload a file using the Files pane.

In [ ]:
OBSERVATION_FILE = Path("observation.npz")
TASK = ""  # Enter the instruction associated with this observation.
assert OBSERVATION_FILE.exists(), "Supply a compatible observation.npz first."
assert TASK.strip(), "Enter the matching task instruction."
arrays = np.load(OBSERVATION_FILE, allow_pickle=False)
frame = {"task": TASK}
for key, feature in policy.config.input_features.items():
    if key not in arrays:
        raise ValueError(f"Missing required feature: {key}")
    value = arrays[key]
    if "images" in key:
        if value.ndim != 3 or value.shape[-1] != 3 or value.dtype != np.uint8:
            raise ValueError(f"{key} must be RGB HxWx3 uint8")
        frame[key] = torch.from_numpy(value.copy()).permute(2, 0, 1).float() / 255.0
    else:
        if tuple(value.shape) != tuple(feature.shape):
            raise ValueError(f"{key}: expected {feature.shape}, got {value.shape}")
        frame[key] = torch.as_tensor(value.copy(), dtype=torch.float32)
print({k: tuple(v.shape) if torch.is_tensor(v) else v for k, v in frame.items()})

## Run genuine inference and save provenance
The processed batch is passed to `select_action`. Reset policy state before an independent observation. Timing includes preprocessing and action postprocessing, but not the initial checkpoint download/load. GPU synchronization prevents reporting only asynchronous launch time. Output dimensions must be interpreted using the source dataset's action definition; they are not the browser demo's image-space arrow.

In [ ]:
policy.reset()
if device == "cuda": torch.cuda.synchronize()
started = time.perf_counter()
batch = preprocess(frame)
with torch.inference_mode():
    action = postprocess(policy.select_action(batch))
if device == "cuda": torch.cuda.synchronize()
elapsed_ms = (time.perf_counter() - started) * 1000
print("Action:", action)
print("Measured inference ms:", elapsed_ms)
record = {
    "schema": "p11-smolvla-observation-v1",
    "model_id": MODEL_ID, "revision": REVISION,
    "device": device, "torch_version": torch.__version__,
    "lerobot_version": importlib.metadata.version("lerobot"),
    "task": TASK, "observation_file": str(OBSERVATION_FILE),
    "input_shapes": {k: list(v.shape) for k, v in frame.items() if torch.is_tensor(v)},
    "action": action.detach().cpu().tolist(), "elapsed_ms": elapsed_ms,
    "interpretation": "Embodiment-specific action; see dataset definitions.",
    "hardware_execution": False, "evaluation_status": "One inference, not task-success evaluation"
}
Path("smolvla_real_inference.json").write_text(json.dumps(record, indent=2))

## Questions to answer in the final project
1. Which features came from a camera, which from language, and which from robot state?
2. What are the units and ordering of each action component?
3. What changed relative to the browser CLIP + deterministic action baseline?
4. Which checkpoint/data mismatch would make the numerical output meaningless?
5. What evaluation would be required before claiming successful robot behavior?

If this notebook fails because of API or observation incompatibility, report that failure. Do not replace its output with an invented trajectory. Only a successfully executed record with the original input provenance may be used as a replay.